# Testy jednostkowe w C++

Laboratorium wprowadzające w testy jednostkowe w C++ z użyciem **GoogleTest** i **gMock**.
Materiał jest podzielony na notebooki — każdy omawia jeden temat i kończy się ćwiczeniem.

## Cele

Po tym laboratorium:

* wiesz, po co pisze się testy jednostkowe i jakie problemy rozwiązują,
* piszesz testy w GoogleTest: asercje, testy sparametryzowane, fixture'y,
* izolujesz testowany kod od zależności za pomocą dublerów (gMock),
* zamieniasz zgłoszenie błędu w test regresyjny i refaktoryzujesz kod pod osłoną testów,
* wiesz, co mierzy pokrycie kodu, a czego nie mierzy, i znasz narzędzia „dla dociekliwych”.

## Jak korzystać z notebooków

* Komórkę uruchamiasz skrótem `Shift+Enter` (albo przyciskiem ▶ na pasku narzędzi).
* Komórki z kodem zaczynające się od `!` wykonują polecenia powłoki — w tym laboratorium
  głównie `cmake` i `ctest`. Każda taka komórka jest niezależna: zaczyna w katalogu notebooka.
* Pliki do edycji otwierasz kliknięciem w link w treści ćwiczenia (otworzą się w edytorze JupyterLab).
  **Zapisz plik** (`Ctrl+S`) przed uruchomieniem komórki, która go kompiluje.

## Wymagania

| Narzędzie | Minimalna wersja |
|---|---|
| Kompilator C++20 | GCC 11, Clang 14, Visual Studio 2022 (MSVC 17), Apple Clang z Xcode 15 |
| CMake | 3.22 |
| Python + JupyterLab | Python 3.10, JupyterLab 4 |
| Pakiety Pythona | `ipykernel` (jądro notebooków) i `gcovr` (raport pokrycia w notebooku 5) — instaluje je `uv sync` z `pyproject.toml` |
| Internet | tylko przy pierwszej kompilacji — CMake pobiera GoogleTest 1.18.0 (raz, do `lab/_deps`) |

Te wersje wynikają z wymagań GoogleTest 1.18 (C++17, [polityka wsparcia Google](https://github.com/google/oss-policies-info/blob/main/foundational-cxx-support-matrix.md)).
Domyślne pakiety w Ubuntu 22.04/24.04 i Debianie 12/13 je spełniają.

## Sprawdzenie środowiska

Poniższe komórki budują mały projekt testowy z katalogu [`_smoke/`](_smoke/smoke_test.cpp).
Używa on tych samych ustawień co wszystkie ćwiczenia (plik [`cmake/lab.cmake`](cmake/lab.cmake)):
standard C++20, GoogleTest i gMock pobrane przez `FetchContent` w przypiętej wersji.

Każde ćwiczenie budujemy tymi samymi trzema krokami:

1. `cmake -S <źródła> -B <katalog build>` — konfiguracja (za pierwszym razem pobiera i konfiguruje GoogleTest),
2. `cmake --build <katalog build>` — kompilacja,
3. `ctest --test-dir <katalog build>` — uruchomienie wszystkich testów.

Opcje `--config Debug` i `-C Debug` są potrzebne tylko w Visual Studio (generator wielokonfiguracyjny);
na Linuksie i macOS nic nie zmieniają.

In [ ]:
# Tool versions
!cmake --version

In [ ]:
# Configure, build and run the smoke test
!cmake -S _smoke -B _smoke/build && cmake --build _smoke/build --config Debug --parallel && ctest --test-dir _smoke/build -C Debug --output-on-failure

Na końcu powinieneś zobaczyć podsumowanie `ctest` z informacją, że przeszły wszystkie testy, np.:

```
100% tests passed, 0 tests failed out of 2
```

(nowsze wersje CMake skracają je do `100% tests passed out of 2`).

Jeśli konfiguracja się nie powiedzie, sprawdź wersję CMake i kompilatora. Komunikat o braku
`<compare>` albo błąd przy `operator<=>` oznacza zbyt stary kompilator.

## Po co testy jednostkowe?

**Test jednostkowy** to mały, automatyczny program, który sprawdza jeden fragment kodu — funkcję
albo klasę — w odizolowaniu od reszty systemu i sam stwierdza, czy wynik jest poprawny
(definicja i jej odmiany: [M. Fowler, *UnitTest*](https://martinfowler.com/bliki/UnitTest.html)).

Jakie problemy rozwiązują testy jednostkowe:

* **Wczesne wykrywanie błędów.** Błąd znaleziony minutę po napisaniu kodu kosztuje minutę. Ten sam błąd
  znaleziony u klienta kosztuje dni pracy, a czasem dużo więcej. Raport NIST z 2002 r. szacował koszty
  niedostatecznego testowania w samych Stanach Zjednoczonych na 22–60 mld dolarów rocznie
  ([NIST Planning Report 02-3](https://www.nist.gov/document/report02-3pdf)).
* **Regresje.** Zmiana w jednym miejscu psuje coś, co wcześniej działało. Zestaw testów uruchamiany
  po każdej zmianie wyłapuje to od razu.
* **Bezpieczna refaktoryzacja.** Można przebudować wnętrze kodu i w kilka sekund sprawdzić, że jego
  zachowanie się nie zmieniło (wrócimy do tego w notebooku 5).
* **Wykonywalna dokumentacja.** Test pokazuje, jak używać funkcji i co ma zwracać w przypadkach brzegowych.
  W przeciwieństwie do komentarza nie może się zdezaktualizować, bo przestałby przechodzić.
* **Lepszy projekt.** Kod, który łatwo przetestować, ma zwykle jasne wejścia i wyjścia oraz jawne zależności.

Gdzie testy jednostkowe są najbardziej przydatne: logika biznesowa, algorytmy, walidacja danych,
obliczenia (szczególnie na wartościach brzegowych), parsowanie, konwersje. Mniej pasują do kodu,
który głównie przekazuje dane dalej (GUI, cienkie warstwy nad bazą danych czy siecią) — tam lepiej
sprawdzają się testy integracyjne.

Pamiętaj też, że testy pokazują obecność błędów, a nie ich brak. Zielony zestaw testów oznacza tylko,
że sprawdzone przypadki działają — dlatego tak ważne jest, *które* przypadki sprawdzamy (notebook 1).

## Kiedy brakuje testów — historie z życia

* **Patriot, 1991.** Zegar systemu obrony przeciwrakietowej liczył czas w dziesiątych częściach sekundy,
  a liczby 0,1 nie da się dokładnie zapisać w systemie dwójkowym. Po 100 godzinach pracy błąd urósł do
  ok. 0,34 s, system nie przechwycił rakiety Scud i zginęło 28 żołnierzy
  ([opis D. N. Arnolda](https://www-users.cse.umn.edu/~arnold/disasters/patriot.html)).
* **Giełda w Vancouver, 1982–1983.** Indeks giełdy po każdej zmianie był *obcinany* (a nie zaokrąglany)
  do trzech miejsc po przecinku, ok. 3000 razy dziennie. Indeks tracił przez to ok. 25 punktów
  miesięcznie; po przeliczeniu w listopadzie 1983 r. skoczył z 524,811 do 1098,892
  ([Wikipedia](https://en.wikipedia.org/wiki/Vancouver_Stock_Exchange)).
* **Ariane 5, 1996.** Kod przeniesiony z Ariane 4 zamieniał 64-bitową liczbę zmiennoprzecinkową na
  16-bitową liczbę całkowitą. W nowej rakiecie wartość się nie mieściła, oprogramowanie zgłosiło błąd
  i rakieta została zniszczona 37 sekund po starcie
  ([raport komisji](https://www-users.cse.umn.edu/~arnold/disasters/ariane5rep.html)).
* **Knight Capital, 2012.** Przy wdrożeniu nowej funkcji ponownie wykorzystano flagę, która na jednym
  z ośmiu serwerów uruchomiła stary, nieużywany kod. Firma straciła ponad 460 mln dolarów
  w ok. 45 minut; starego kodu nie przetestowano ponownie po wcześniejszych zmianach
  ([decyzja SEC, PDF](https://www.sec.gov/litigation/admin/2013/34-70694.pdf)).
* **„goto fail”, Apple, 2014.** Zdublowana linia `goto fail;` w C sprawiła, że weryfikacja podpisu
  w TLS była pomijana ([analiza A. Langleya](https://www.imperialviolet.org/2014/02/22/applebug.html)).

Dwie pierwsze historie dotyczą reprezentacji liczb — wrócimy do tego tematu w notebooku 1.

## Piramida testów

<img src="_common/images/test_pyramid.svg" alt="Piramida testów" width="560">

Testy różnią się zakresem. **Testy jednostkowe** sprawdzają pojedyncze funkcje i klasy — są małe,
odizolowane i szybkie, więc może ich być bardzo dużo. **Testy integracyjne** sprawdzają współpracę
kilku komponentów (np. kod + baza danych). **Testy end-to-end (E2E) / UI** sprawdzają cały system
tak, jak używa go użytkownik — są najbardziej realistyczne, ale wolne, kosztowne i kruche.

Piramida (Mike Cohn, *Succeeding with Agile*, 2009) podsumowuje dwie zasady
([Ham Vocke, *The Practical Test Pyramid*](https://martinfowler.com/articles/practical-test-pyramid.html)):
pisz testy o różnej szczegółowości, a im wyższy poziom, tym mniej testów.
W polskim sylabusie ISTQB najniższa warstwa nazywa się **testami modułowymi**
([sylabus ISTQB CTFL 4.0, §5.1.6](https://sjsi.org/download/13814/)).

Piramida to model, a nie prawo. W aplikacjach webowych popularny jest np. „testing trophy”
z większym udziałem testów integracyjnych
([K. C. Dodds](https://kentcdodds.com/blog/the-testing-trophy-and-testing-classifications)).
M. Fowler zauważa, że spory o kształt wynikają głównie z różnych definicji „testu jednostkowego”
([*On the Diverse And Fantastical Shapes of Testing*](https://martinfowler.com/articles/2021-test-shapes.html)).
W tym laboratorium zajmujemy się podstawą piramidy.

## Budowa testu: Arrange – Act – Assert

Dobrze napisany test ma trzy wyraźne części
([B. Wake, *3A – Arrange, Act, Assert*](https://xp123.com/3a-arrange-act-assert/)):

1. **Arrange** — przygotuj obiekt i dane,
2. **Act** — wykonaj jedną testowaną operację,
3. **Assert** — sprawdź wynik.

```cpp
#include <string>
#include <vector>

#include <gtest/gtest.h>

TEST(VectorTest, PushBackIncreasesSize) {
    // Arrange
    std::vector<std::string> names{"Ala", "Ola"};

    // Act
    names.push_back("Ela");

    // Assert
    EXPECT_EQ(names.size(), 3u);
    EXPECT_EQ(names.back(), "Ela");
}
```

Makra `TEST` i `EXPECT_EQ` omawiamy w notebooku 1. G. Meszaros opisuje ten sam schemat jako
*Four-Phase Test* — z czwartą fazą, sprzątaniem
([xUnit Patterns](http://xunitpatterns.com/Four%20Phase%20Test.html)); do sprzątania wrócimy w notebooku 3.
W podejściu BDD te same części nazywają się *Given – When – Then*.

## Cechy dobrego testu: FIRST

Akronim FIRST ([T. Ottinger, J. Langr, *Agile in a Flash*](https://agileinaflash.blogspot.com/2009/02/first.html))
zbiera cechy dobrych testów jednostkowych:

| Litera | Cecha | Co to znaczy |
|---|---|---|
| **F** | *Fast* — szybki | Test trwa milisekundy, więc cały zestaw można uruchamiać po każdej zmianie. |
| **I** | *Independent / Isolated* — niezależny | Testy nie zależą od siebie nawzajem ani od kolejności uruchamiania; każdy przygotowuje własny stan. |
| **R** | *Repeatable* — powtarzalny | Test daje ten sam wynik za każdym razem i w każdym środowisku. Nie zależy od sieci, zewnętrznych usług, plików na dysku ani od losowości. |
| **S** | *Self-validating* — samosprawdzający | Test sam rozstrzyga: przeszedł albo nie. Nikt nie musi czytać wydruku i oceniać wyniku. |
| **T** | *Timely* — na czas | Test powstaje w odpowiednim momencie — tuż przed kodem albo razem z nim, a nie „kiedyś”. |

W oryginale „I” oznacza *Isolated*; w książce *Clean Code* R. C. Martina ta sama zasada nazywa się
*Independent*. Obie nazwy znaczą to samo.

Częstą przyczyną łamania zasady **R** są zależności od świata zewnętrznego: test, który łączy się
z prawdziwym serwerem, czyta plik z dysku albo losuje dane bez ustalonego ziarna, raz przechodzi,
a raz nie ([M. Fowler, *Eradicating Non-Determinism in Tests*](https://martinfowler.com/articles/nonDeterminism.html)).
W testach takie zależności zastępuje się dublerami — pokażemy to w notebooku 4.

## Plan laboratorium

1. [Podstawy GoogleTest](01_basics/01_basics.ipynb)
2. [Testy sparametryzowane](02_parametrized/02_parametrized.ipynb)
3. [Fixture'y i cykl życia testu](03_fixtures/03_fixtures.ipynb)
4. [Dublery testowe i gMock](04_mocks/04_mocks.ipynb)
5. [Jakość testów](05_quality/05_quality.ipynb)
6. [Ciekawostki dla dociekliwych](06_extras/06_extras.ipynb)

## Linki i materiały

* [GoogleTest — Primer](https://google.github.io/googletest/primer.html) — oficjalne wprowadzenie
* [GoogleTest — Quickstart: CMake](https://google.github.io/googletest/quickstart-cmake.html)
* [Sylabus ISTQB CTFL 4.0 (PL, PDF)](https://sjsi.org/download/13814/) — rozdz. 1.2 „Dlaczego testowanie jest niezbędne?”
* [M. Fowler — *TestPyramid*](https://martinfowler.com/bliki/TestPyramid.html)
* [D. Steffen — *Back to Basics: Unit Testing in C++*, CppCon 2024 (wideo)](https://www.youtube.com/watch?v=MwoAM3sznS0)
* [P. Nash — *Back to Basics: Testing in C++*, CppCon 2023 (wideo)](https://www.youtube.com/watch?v=7_H4qzhWbnQ)